# Optimización y Despliegue de Modelos Predictivos Supervisados - Eric Rodriguez
### Plataforma de Formación Continua — Cierre del módulo

**Contexto:** el equipo de analítica debe automatizar dos decisiones:
1. Predecir si un estudiante **completará** un curso online.
2. Estimar cuántos **cursos futuros** tomará durante el próximo trimestre.

Ambos modelos se optimizan con `GridSearchCV` y se validan con `cross_validate`, dentro de un `Pipeline` con buenas prácticas de escalamiento y codificación.

## 0. Carga y exploración del dataset

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import cross_validate, GridSearchCV, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

pd.set_option('display.max_columns', None)

df = pd.read_csv('optimizacion_modelos_predictivos.csv')
print(f"Dataset: {df.shape[0]} filas x {df.shape[1]} columnas")
df


Dataset: 10 filas x 7 columnas


,edad,nivel_participacion,tiempo_en_plataforma,tipo_inscripcion,region,completo,cursos_futuros
0,24,0.85,10,Libre,Sur,1,3
1,31,0.60,6,Premium,Centro,0,1
2,29,0.90,15,Premium,Centro,1,4
3,40,0.50,5,Libre,Norte,0,0
4,27,0.78,8,Libre,Sur,1,2
5,35,0.65,7,Premium,Norte,1,3
6,50,0.35,3,Libre,Centro,0,1
7,22,0.95,20,Premium,Sur,1,5
8,38,0.55,4,Libre,Norte,0,0
9,28,0.88,12,Premium,Centro,1,4


In [4]:
df.info()
print("\nValores nulos por columna:")
print(df.isnull().sum())
print("\nDistribución de 'completo' (target de clasificación):")
print(df['completo'].value_counts())
print("\nDistribución de 'cursos_futuros' (target de regresión):")
print(df['cursos_futuros'].describe())


<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   edad                  10 non-null     int64  
 1   nivel_participacion   10 non-null     float64
 2   tiempo_en_plataforma  10 non-null     int64  
 3   tipo_inscripcion      10 non-null     str    
 4   region                10 non-null     str    
 5   completo              10 non-null     int64  
 6   cursos_futuros        10 non-null     int64  
dtypes: float64(1), int64(4), str(2)
memory usage: 692.0 bytes

Valores nulos por columna:
edad                    0
nivel_participacion     0
tiempo_en_plataforma    0
tipo_inscripcion        0
region                  0
completo                0
cursos_futuros          0
dtype: int64

Distribución de 'completo' (target de clasificación):
completo
1    6
0    4
Name: count, dtype: int64

Distribución de 'cursos_futuros' (target de regresión):
coun

Variables
- Objetivo de clasificación: `completo` (binaria: 0/1).
- Objetivo de regresión: `cursos_futuros` (numérica de conteo).
- Predictores (comunes a ambos modelos): `edad`, `nivel_participacion`, `tiempo_en_plataforma` (numéricas) y `tipo_inscripcion`, `region` (categóricas).

ni `completo` se usa como predictor de `cursos_futuros`, ni viceversa, porque ambas variables describen resultados del mismo período futuro (evita *data leakage*).

In [6]:
variables_numericas = ['edad', 'nivel_participacion', 'tiempo_en_plataforma']
variables_categoricas = ['tipo_inscripcion', 'region']

X = df[variables_numericas + variables_categoricas]
y_clasificacion = df['completo']
y_regresion = df['cursos_futuros']

preprocesador = ColumnTransformer(transformers=[
    ('num', StandardScaler(), variables_numericas),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), variables_categoricas)
])


## 1. Modelo de clasificación con GridSearchCV

**Algoritmo elegido:** `RandomForestClassifier`.

**Hiperparámetros a optimizar:** `n_estimators` (cantidad de árboles) y `max_depth` (profundidad máxima), dos de los hiperparámetros más influyentes en el balance sesgo-varianza de un Random Forest.

**Validación:** dado el tamaño extremadamente reducido del dataset (n=10) y el leve desbalance de clases (6 vs. 4), se usa `StratifiedKFold` tanto para la búsqueda de hiperparámetros como para la validación cruzada final, con un número de folds deliberadamente bajo (`cv=2`) porque con más folds algunas particiones quedarían con 1-2 observaciones, lo cual haría el proceso aún menos confiable.

In [7]:
pipeline_clasificacion = Pipeline(steps=[
    ('preprocesamiento', preprocesador),
    ('modelo', RandomForestClassifier(random_state=42))
])

grid_parametros_clasificacion = {
    'modelo__n_estimators': [50, 100, 200],
    'modelo__max_depth': [2, 4, None]
}

cv_interno = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

grid_search_clasificacion = GridSearchCV(
    pipeline_clasificacion,
    param_grid=grid_parametros_clasificacion,
    cv=cv_interno,
    scoring='f1_macro'
)

grid_search_clasificacion.fit(X, y_clasificacion)

print("Mejores hiperparámetros encontrados (clasificación):")
print(grid_search_clasificacion.best_params_)
print(f"Mejor score (f1_macro) durante la búsqueda: {grid_search_clasificacion.best_score_:.4f}")


Mejores hiperparámetros encontrados (clasificación):
{'modelo__max_depth': 2, 'modelo__n_estimators': 50}
Mejor score (f1_macro) durante la búsqueda: 0.8810


In [8]:
mejor_pipeline_clasificacion = grid_search_clasificacion.best_estimator_

cv_externo = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

resultados_clasificacion = cross_validate(
    mejor_pipeline_clasificacion,
    X, y_clasificacion,
    cv=cv_externo,
    scoring=['accuracy', 'f1_macro']
)

print("=== Validación cruzada del mejor modelo (clasificación, 3-fold estratificado) ===")
print(f"Accuracy por fold: {resultados_clasificacion['test_accuracy'].round(3)}")
print(f"F1-macro por fold: {resultados_clasificacion['test_f1_macro'].round(3)}")
print(f"\nAccuracy promedio: {resultados_clasificacion['test_accuracy'].mean():.4f}")
print(f"F1-macro promedio: {resultados_clasificacion['test_f1_macro'].mean():.4f}")


=== Validación cruzada del mejor modelo (clasificación, 3-fold estratificado) ===
Accuracy por fold: [0.75  0.667 1.   ]
F1-macro por fold: [0.733 0.667 1.   ]

Accuracy promedio: 0.8056
F1-macro promedio: 0.8000


**Interpretación técnica breve:**

Los mejores hiperparámetros y las métricas obtenidas se muestran arriba. Es fundamental notar la alta variabilidad entre folds: con solo 10 observaciones repartidas en 3 particiones , un solo caso mal clasificado puede mover el accuracy de un fold en 25-33 puntos porcentuales.

## 2. Modelo de regresión con GridSearchCV 

Algoritmo elegido: `RandomForestRegressor`.

se mantiene el mismo tipo de algoritmo de base  que en clasificación por consistencia metodológica y porque no se asume una relación lineal entre las variables predictoras y `cursos_futuros`. Se descartó `Ridge` como alternativa porque, con solo 5 predictores originales (2 numéricos + 3 dummies de las categóricas), el problema no presenta el tipo de alta dimensionalidad/multicolinealidad que la regularización de Ridge está pensada para resolver, y un modelo de árboles permite capturar mejor posibles relaciones no lineales pese al tamaño reducido de los datos.
Hiperparámetros a optimizar: `n_estimators` y `max_depth` 
Validación:`KFold` simple con `cv=2`

In [9]:
pipeline_regresion = Pipeline(steps=[
    ('preprocesamiento', preprocesador),
    ('modelo', RandomForestRegressor(random_state=42))
])

grid_parametros_regresion = {
    'modelo__n_estimators': [50, 100, 200],
    'modelo__max_depth': [2, 4, None]
}

cv_interno_regresion = KFold(n_splits=2, shuffle=True, random_state=42)

grid_search_regresion = GridSearchCV(
    pipeline_regresion,
    param_grid=grid_parametros_regresion,
    cv=cv_interno_regresion,
    scoring='r2'
)

grid_search_regresion.fit(X, y_regresion)

print("Mejores hiperparámetros encontrados (regresión):")
print(grid_search_regresion.best_params_)
print(f"Mejor score (r2) durante la búsqueda: {grid_search_regresion.best_score_:.4f}")


Mejores hiperparámetros encontrados (regresión):
{'modelo__max_depth': 4, 'modelo__n_estimators': 50}
Mejor score (r2) durante la búsqueda: 0.7222


In [10]:
mejor_pipeline_regresion = grid_search_regresion.best_estimator_

cv_externo_regresion = KFold(n_splits=3, shuffle=True, random_state=42)

resultados_regresion = cross_validate(
    mejor_pipeline_regresion,
    X, y_regresion,
    cv=cv_externo_regresion,
    scoring=['neg_mean_absolute_error', 'r2']
)

mae_promedio = -resultados_regresion['test_neg_mean_absolute_error'].mean()
r2_promedio = resultados_regresion['test_r2'].mean()

print("=== Validación cruzada del mejor modelo (regresión, 3-fold) ===")
print(f"MAE por fold: {(-resultados_regresion['test_neg_mean_absolute_error']).round(3)}")
print(f"R2 por fold: {resultados_regresion['test_r2'].round(3)}")
print(f"\nMAE promedio: {mae_promedio:.4f} cursos")
print(f"R2 promedio: {r2_promedio:.4f}")


=== Validación cruzada del mejor modelo (regresión, 3-fold) ===
MAE por fold: [0.78  1.753 0.593]
R2 por fold: [  0.563 -13.285   0.41 ]

MAE promedio: 1.0422 cursos
R2 promedio: -4.1041


**Justificación de las decisiones técnicas tomadas:**

Se reporta `neg_mean_absolute_error` por ser una métrica directamente interpretable en la unidad de negocio ("cursos"), y `r2` para cuantificar el poder explicativo del modelo. Con `cv=3` sobre solo 10 datos, cada fold de test contiene apenas 3-4 observaciones: el R² calculado en folds tan pequeños es numéricamente inestable.

## 3. Reflexión integradora y análisis de desempeño 

Resumen de resultados obtenidos:

| Modelo | Mejores hiperparámetros | Métrica 1 | Métrica 2 | Estabilidad entre folds |
|---|---|---|---|---|
| Clasificación (RandomForestClassifier) | max_depth=2, n_estimators=50 | Accuracy = 0.806 | F1-macro = 0.800 | Folds: [0.75, 0.67, 1.00] — variable pero siempre positivo |
| Regresión (RandomForestRegressor) | max_depth=4, n_estimators=50 | MAE = 1.042 cursos | **R² = -4.104** | Folds: [0.56, **-13.29**, 0.41] — inestable |

### Comparación de enfoques entre clasificación y regresión

Ambos modelos comparten exactamente la misma arquitectura metodológica: un `Pipeline` con el mismo `ColumnTransformer` (escalamiento + codificación), el mismo algoritmo base (Random Forest) y la misma estrategia de búsqueda de hiperparámetros (`GridSearchCV` sobre `n_estimators` y `max_depth`). La diferencia de enfoque real estuvo en la métrica de optimización usada dentro del `GridSearchCV` (`f1_macro` para clasificación, `r2` para regresión, cada una apropiada al tipo de variable objetivo) y en el tipo de partición de validación.

### Cómo influyeron la validación cruzada y el ajuste de hiperparámetros en el rendimiento final

La validación cruzada expuso un problema que un solo train/test split habría ocultado: el modelo de regresión obtuvo R²=0.7222 durante la búsqueda de hiperparámetros (cv=2), pero al validarlo con `cross_validate` en cv=3, un fold arrojó **R²=-13.285** — un valor negativo extremo, esperable al calcular R² sobre folds de solo 3-4 observaciones, donde un solo dato atípico distorsiona la métrica. El ajuste de hiperparámetros optimizó parámetros técnicamente válidos, pero **no puede compensar la falta de datos**: 10 observaciones son insuficientes para validar un modelo de regresión con confianza.

### Impacto de un buen preprocesamiento en el éxito del modelo

El `Pipeline` con `ColumnTransformer` evitó *data leakage* al ajustar el preprocesamiento solo con los datos de entrenamiento de cada fold — buena práctica respetada en ambos modelos. Pero esto deja una lección clara: un buen preprocesamiento es **necesario pero no suficiente**. Ningún pipeline, por correcto que sea, compensa una muestra insuficiente; el éxito de un modelo depende tanto del proceso técnico como de la cantidad de datos disponibles, y esto último fue la limitación dominante aquí.

### ¿Cuál de los dos modelos es más confiable, y por qué?

El de **clasificación**, por dos razones: (1) sus métricas se mantuvieron en un rango razonable y positivo entre folds (0.67 a 1.00), mientras la regresión produjo un R² muy negativo en un fold; y (2) accuracy/f1 están acotadas entre 0 y 1, limitando el impacto de un fold "malo", mientras el R² no tiene limite y puede volverse arbitrariamente negativo (promedio final: -4.10, inutilizable). En la práctica: el modelo de clasificación es un punto de partida razonabl, el de regresión **no debería desplegarse** en su estado actual — la prioridad es conseguir más datos, no seguir ajustando hiperparámetros y generar un procedimiento mas robusto

